## Методические указания по выполнению практикума №6

[МУ блока](README.md) · [Общие МУ](../../../docs/guidelines-students.md) · [Рубрика оценивания](../teachers-assessment/README.md)

**Тема: Разработка стратегий применения базисных (foundation) моделей компьютерного зрения: zero-shot learning, linear и non-linear probe**

**Тема РПД:** Л16/П6. **Индикатор:** DL-3.2, уровень П.

**Ноутбук:** `#8 P6_Foundation_models.ipynb`. Практикум завершает блок 3: вместо обучения модели под задачу исследуется, какую часть задачи можно решить **без обучения вообще** и сколько разметки требуется, чтобы обучение окупилось.

**Цель работы:** сравнить три стратегии применения базисной модели к целевой задаче классификации — zero-shot по текстовым запросам, linear probe и non-linear probe поверх замороженных признаков — и обосновать выбор стратегии по качеству, объёму требуемой разметки и вычислительной стоимости.

**Задачи:**

- Разобрать принцип контрастного мультимодального обучения и механику zero-shot-классификации через текстовые запросы.
- Реализовать zero-shot-классификацию на CLIP и исследовать чувствительность результата к формулировке запроса.
- Извлечь и закэшировать эмбеддинги замороженного энкодера изображений.
- Обучить linear probe и non-linear probe по единому протоколу.
- Построить few-shot-кривую: качество как функция числа размеченных примеров на класс.
- Свести стратегии в сопоставимую таблицу и сформулировать правило выбора стратегии под бюджет разметки.

### 1. Теоретическая часть

#### 1.1 Базисные модели

Базисная (foundation) модель обучается на большом неразмеченном или слабо размеченном корпусе и предоставляет универсальное представление, переиспользуемое во множестве задач без обучения с нуля. В компьютерном зрении сложились три линии:

- **Самообучение по маскированию:** MAE, I-JEPA — модель восстанавливает скрытую часть изображения.
- **Контрастное самообучение:** SimCLR, DINO, DINOv2 — сближаются представления разных аугментаций одного изображения.
- **Мультимодальное контрастное обучение:** CLIP, ALIGN — сближаются представления изображения и его текстового описания.

Отдельно стоят промптируемые модели сегментации и детекции (SAM, Grounding DINO) и мультимодальные языковые модели (VLM).

Практически значимо то, что мультимодальное обучение даёт **общее пространство** изображений и текста. Это позволяет задавать классы текстом на этапе применения, а не на этапе обучения.

#### 1.2 CLIP и zero-shot-классификация

CLIP обучается на парах «изображение — текст». Энкодеры $f_I$ и $f_T$ отображают изображение и текст в векторы одной размерности, которые нормируются: $\hat{u} = u / \lVert u \rVert$. Обучение минимизирует симметричный контрастный (InfoNCE) функционал; для направления «изображение → текст» в батче размера $N$:

$$\mathcal{L}_{I \to T} = -\frac{1}{N}\sum_{i=1}^{N} \log \frac{\exp\!\left(\langle \hat{u}_i, \hat{v}_i \rangle / \tau\right)}{\sum_{j=1}^{N} \exp\!\left(\langle \hat{u}_i, \hat{v}_j \rangle / \tau\right)},$$

где $\hat{u}_i$ — эмбеддинг изображения, $\hat{v}_j$ — эмбеддинг текста, $\tau$ — обучаемая температура. Совпадающая пара — положительный пример, все остальные пары в батче — отрицательные.

**Zero-shot-классификация.** Каждый класс $c$ описывается текстовым запросом $t_c$ (например, `"a photo of a Beagle, a type of pet"`). Предсказание:

$$\hat{y}(x) = \arg\max_{c} \; \langle \hat{u}(x), \hat{v}(t_c) \rangle .$$

Обучаемых параметров нет; вся «настройка» — формулировка запросов. Отсюда два следствия:

1. Качество зависит от формулировки. Слово `"photo"`, добавление домена (`"a type of pet"`), единственное или множественное число — всё это измеримо меняет результат. Поэтому zero-shot без указания использованных шаблонов невоспроизводим.
2. **Ансамблирование шаблонов**: для каждого класса берётся несколько шаблонов, их нормированные текстовые эмбеддинги усредняются, результат нормируется повторно:

$$\hat{v}_c = \frac{\frac{1}{K}\sum_{k=1}^{K} \hat{v}(t_{c,k})}{\left\lVert \frac{1}{K}\sum_{k=1}^{K} \hat{v}(t_{c,k}) \right\rVert}.$$

Важно: выбор шаблонов — это подбор гиперпараметра. Он ведётся по validation. Перебор шаблонов по test превращает «zero-shot» в скрытую настройку по тесту и делает оценку недостоверной.

#### 1.3 Linear probe и non-linear probe

**Probe** — диагностический классификатор поверх замороженных признаков. Backbone не обучается, эмбеддинги извлекаются один раз и переиспользуются.

- **Linear probe:** $\hat{y} = \mathrm{softmax}(W \hat{u} + b)$. Обучаемых параметров $D \times C + C$. Измеряет **линейную разделимость** классов в пространстве признаков: высокое качество означает, что нужная информация в представлении уже присутствует в явном виде.
- **Non-linear probe:** одна скрытая нелинейность, $\hat{y} = \mathrm{softmax}(W_2\,\sigma(W_1 \hat{u} + b_1) + b_2)$. Прирост относительно linear probe означает, что информация в представлении есть, но закодирована нелинейно. Отсутствие прироста при большем числе параметров — тоже результат: он говорит, что упирается задача не в мощность головы.

Probe принципиально отличается от fine-tuning: backbone заморожен, поэтому стоимость обучения на порядки ниже, а риск переобучения на малой выборке существенно меньше. Сравнение linear probe и частичного fine-tuning выполняется в ЛР блока 4; здесь предмет — граница между «не обучать вообще» и «обучить лёгкую голову».

#### 1.4 Выбор стратегии под бюджет разметки

| Условие | Стратегия | Стоимость |
|---|---|---|
| разметки нет, классы описуемы словами | zero-shot | только инференс |
| несколько примеров на класс | few-shot linear probe | минуты на CPU/GPU |
| десятки–сотни примеров на класс | linear / non-linear probe | сопоставима с probe |
| тысячи примеров, домен далёк от предобучения | fine-tuning backbone | часы GPU |

Практический вопрос, на который отвечает эта работа: **начиная с какого числа размеченных примеров на класс probe превосходит zero-shot**. Точка пересечения few-shot-кривой с уровнем zero-shot — количественная характеристика задачи, а не универсальная константа.

#### 1.5 Методика сопоставимого сравнения

Все стратегии сравниваются при: одном разбиении данных, одном наборе классов, одном энкодере изображений и одном препроцессинге, одном seed, одном устройстве. Различие между сериями — только стратегия (и, в few-shot-серии, число примеров на класс). Гиперпараметры probe и набор шаблонов выбираются по validation; test используется один раз.

### 2. Практическая часть

#### 2.1 Подготовка окружения

In [ ]:
# Версии закреплены по мажорной компоненте: незакреплённая установка ломает воспроизводимость.
%pip install -q "torch>=2.2,<3.0" "torchvision>=0.17,<1.0" "transformers>=4.44,<5.0" "timm>=1.0,<2.0" "scikit-learn>=1.4,<2.0" "pandas>=2.0,<3.0" "matplotlib>=3.8,<4.0"

In [ ]:
import json
import random
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

import torchvision
import transformers
from torchvision.datasets import OxfordIIITPet
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

SEED = 42
DATA_ROOT = Path("data")
OUTPUT_DIR = Path("outputs")
(OUTPUT_DIR / "embeddings").mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", DEVICE)
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("transformers:", transformers.__version__)

In [ ]:
def set_global_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


RUNS_PATH = OUTPUT_DIR / "runs.jsonl"
RUNS: list = []


def log_run(name: str, **fields) -> dict:
    """Записать стратегию и её результат в журнал экспериментов."""
    record = {"name": name, "seed": SEED, "device": DEVICE.type, **fields}
    RUNS.append(record)
    with RUNS_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return record


def results_table(columns=None) -> pd.DataFrame:
    df = pd.DataFrame(RUNS)
    if columns:
        columns = [c for c in columns if c in df.columns]
        df = df[columns]
    return df


def evaluate_predictions(y_true, y_pred) -> dict:
    """Единый набор метрик для всех стратегий работы."""
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


set_global_seed()

#### 2.2 Данные

Используется подмножество Oxford-IIIT Pet из 10 пород. Небольшое число классов выбрано сознательно: (1) для zero-shot нужны классы, которые можно описать естественным текстом, и результат должен быть интерпретируем вручную; (2) 10 классов дают читаемую confusion matrix; (3) при малом числе классов few-shot-кривая строится за приемлемое время.

**Параметр `N_SUBSET`** дополнительно ограничивает объём данных. Он нужен, чтобы извлечение эмбеддингов и вся серия probe укладывались в 2 академических часа на одном бесплатном GPU, и чтобы все стратегии сравнивались при одинаковом бюджете данных. При изменении `N_SUBSET` абсолютные значения метрик изменятся — это ограничение указывается в выводе.

Метки классов переиндексируются в диапазон `0..9`, исходные имена сохраняются: они понадобятся для построения текстовых запросов.

In [ ]:
SUBSET_CLASSES = [
    "Beagle", "Bengal", "Boxer", "British_Shorthair", "Chihuahua",
    "Maine_Coon", "Persian", "Pug", "Siamese", "Sphynx",
]
N_SUBSET = 600       # изображений в train (по 60 на класс)
N_VAL = 300
N_TEST = 300
BATCH_SIZE = 32

trainval_raw = OxfordIIITPet(root=DATA_ROOT, split="trainval",
                             target_types="category", download=True)
test_raw = OxfordIIITPet(root=DATA_ROOT, split="test",
                         target_types="category", download=True)

ALL_CLASSES = list(trainval_raw.classes)
missing = [name for name in SUBSET_CLASSES if name not in ALL_CLASSES]
assert not missing, f"классы отсутствуют в датасете: {missing}"

ORIGINAL_IDS = [ALL_CLASSES.index(name) for name in SUBSET_CLASSES]
REMAP = {original: new for new, original in enumerate(ORIGINAL_IDS)}
NUM_CLASSES = len(SUBSET_CLASSES)
CLASS_NAMES = [name.replace("_", " ") for name in SUBSET_CLASSES]
print("классы:", CLASS_NAMES)


def dataset_labels(dataset) -> np.ndarray:
    for attr in ("_labels", "targets", "labels"):
        values = getattr(dataset, attr, None)
        if values is not None:
            return np.asarray(values)
    return np.asarray([dataset[i][1] for i in range(len(dataset))])


def select_indices(dataset, n_total: int, seed: int, exclude=None):
    """Стратифицированный отбор индексов только по классам из SUBSET_CLASSES."""
    labels = dataset_labels(dataset)
    rng = np.random.default_rng(seed)
    exclude = exclude or set()
    per_class = max(1, n_total // NUM_CLASSES)
    chosen = []
    for original_id in ORIGINAL_IDS:
        pool = [i for i in np.flatnonzero(labels == original_id) if i not in exclude]
        rng.shuffle(pool)
        chosen.extend(int(i) for i in pool[:per_class])
    rng.shuffle(chosen)
    return chosen[:n_total]


train_idx = select_indices(trainval_raw, N_SUBSET, SEED)
val_idx = select_indices(trainval_raw, N_VAL, SEED + 1, exclude=set(train_idx))
test_idx = select_indices(test_raw, N_TEST, SEED + 2)
assert not (set(train_idx) & set(val_idx)), "train и validation пересекаются"

SPLITS = {"train": (trainval_raw, train_idx), "val": (trainval_raw, val_idx),
          "test": (test_raw, test_idx)}
print({name: len(indices) for name, (_, indices) in SPLITS.items()})

# TODO (задание 1.1): проверьте, что в каждом сплите представлены все 10 классов
# и распределение близко к равномерному. Приведите таблицу в отчёте.

#### 2.3 Загрузка CLIP

`CLIPModel` содержит оба энкодера, `CLIPProcessor` — препроцессинг изображений и токенизацию текста. Используемые методы:

- `processor(images=..., return_tensors="pt")` и `processor(text=..., padding=True, return_tensors="pt")`;
- `model.get_image_features(**image_inputs)` → `[B, D]`;
- `model.get_text_features(**text_inputs)` → `[C, D]`;
- `model(**inputs).logits_per_image` — готовые логиты сходства с учётом обучаемой температуры.

Эмбеддинги, возвращаемые `get_*_features`, **не нормированы**: перед вычислением косинусной близости нормировать обязательно.

In [ ]:
CLIP_CHECKPOINT = "openai/clip-vit-base-patch32"

clip_model = CLIPModel.from_pretrained(CLIP_CHECKPOINT).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_CHECKPOINT)

EMBEDDING_DIM = clip_model.config.projection_dim
print("размерность общего пространства:", EMBEDDING_DIM)
print("параметров в модели:", sum(p.numel() for p in clip_model.parameters()))
print("logit_scale:", float(clip_model.logit_scale.exp()))

for parameter in clip_model.parameters():
    parameter.requires_grad_(False)   # backbone заморожен во всей работе

#### 2.4 Zero-shot-классификация

Шаблоны запросов заданы ниже тремя наборами. Набор `minimal` — только имя класса; `photo` — классический шаблон CLIP; `domain` — шаблон с указанием домена, снимающий часть неоднозначности (например, `"Boxer"` без контекста — это ещё и боксёр-спортсмен).

Функция построения текстовых эмбеддингов реализуется вами: в ней сосредоточены все технические тонкости (нормировка, усреднение по шаблонам, повторная нормировка).

In [ ]:
PROMPT_SETS = {
    "minimal": ["{}"],
    "photo": ["a photo of a {}."],
    "domain": [
        "a photo of a {}, a type of pet.",
        "a close-up photo of a {}.",
        "a photo of the pet breed {}.",
    ],
}


@torch.no_grad()
def build_text_embeddings(class_names, templates) -> torch.Tensor:
    """Построить матрицу текстовых эмбеддингов классов [C, D].

    Контракт:
        class_names -- имена классов в порядке индексов 0..C-1;
        templates   -- список шаблонов с одним слотом '{}' под имя класса;
        возвращает L2-нормированный тензор [C, D] на DEVICE.

    Требования:
        * для каждого класса строятся все шаблоны;
        * эмбеддинги шаблонов нормируются ДО усреднения;
        * усреднённый вектор нормируется ПОВТОРНО.
    Порядок нормировок существенен: усреднение ненормированных векторов
    даёт вес пропорционально длине вектора, а не смыслу.
    """
    raise NotImplementedError


@torch.no_grad()
def extract_image_features(split_name: str, batch_size: int = BATCH_SIZE):
    """Извлечь L2-нормированные эмбеддинги изображений сплита.

    Возвращает (features [N, D] на CPU, labels [N] после переиндексации,
    время извлечения в секундах). Эмбеддинги извлекаются ОДИН раз и
    переиспользуются всеми стратегиями -- именно это делает probe дешёвым.
    """
    dataset, indices = SPLITS[split_name]
    features, labels = [], []
    started = time.perf_counter()
    for start in range(0, len(indices), batch_size):
        batch_indices = indices[start:start + batch_size]
        images, targets = [], []
        for index in batch_indices:
            image, target = dataset[index]
            images.append(image.convert("RGB"))
            targets.append(REMAP[int(target)])
        inputs = clip_processor(images=images, return_tensors="pt").to(DEVICE)
        embeddings = clip_model.get_image_features(**inputs)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        features.append(embeddings.cpu())
        labels.extend(targets)
    return torch.cat(features), np.asarray(labels), time.perf_counter() - started

In [ ]:
FEATURES = {}
LABELS = {}
FEATURE_TIME = {}

for split_name in ("train", "val", "test"):
    FEATURES[split_name], LABELS[split_name], FEATURE_TIME[split_name] = \
        extract_image_features(split_name)
    print(f"{split_name}: {tuple(FEATURES[split_name].shape)}, "
          f"{FEATURE_TIME[split_name]:.1f} c")

np.savez_compressed(
    OUTPUT_DIR / "embeddings" / "clip_features.npz",
    **{f"{name}_x": FEATURES[name].numpy() for name in FEATURES},
    **{f"{name}_y": LABELS[name] for name in LABELS},
)


def zero_shot_predict(features: torch.Tensor, text_embeddings: torch.Tensor) -> np.ndarray:
    """Предсказать классы по максимуму косинусной близости.

    Обе матрицы уже L2-нормированы, поэтому косинусная близость -- это
    обычное матричное произведение features @ text_embeddings.T.
    """
    raise NotImplementedError


# TODO (задание 2.1): реализуйте build_text_embeddings и zero_shot_predict.
# TODO (задание 2.2): убедитесь, что нормировка выполнена: норма каждой строки
# возвращаемой матрицы должна быть равна 1 с точностью до 1e-5.

**Задание 3: чувствительность к формулировке запроса.**

Оцените zero-shot на **validation** для всех трёх наборов шаблонов, выберите лучший набор и только после этого примените его к test — один раз. Перебор шаблонов по test означал бы настройку по тесту при формально «нулевом обучении».

In [ ]:
# TODO (задание 3.1): для каждого набора шаблонов из PROMPT_SETS
# постройте текстовые эмбеддинги, предскажите классы на VALIDATION,
# посчитайте метрики через evaluate_predictions и запишите результат
# в журнал через log_run со стратегией "zero_shot" и полем prompt_set.

# TODO (задание 3.2): выберите лучший набор по validation accuracy,
# зафиксируйте его в переменной BEST_PROMPT_SET и оцените на TEST один раз.

BEST_PROMPT_SET = ""   # TODO

# TODO (задание 3.3): приведите в отчёте разброс accuracy между наборами шаблонов.
# Ответьте: можно ли считать zero-shot «методом без гиперпараметров»?

# TODO (задание 3.4) (необязательно): предложите четвёртый набор шаблонов,
# улучшающий результат, и объясните, какую неоднозначность он снимает.

#### 2.5 Linear probe

Голова обучается поверх уже извлечённых эмбеддингов, поэтому обучение занимает секунды: изображения через энкодер повторно не прогоняются.

Протокол обучения фиксируется здесь и применяется без изменений ко всем probe-конфигурациям: тот же оптимизатор, то же число эпох, тот же критерий выбора лучшего состояния (validation accuracy). Иначе прирост non-linear probe будет объясняться не архитектурой головы, а разным бюджетом обучения.

In [ ]:
@dataclass
class ProbeConfig:
    name: str
    head: str = "linear"          # linear | mlp
    hidden_dim: int = 512
    dropout: float = 0.0
    epochs: int = 30
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    batch_size: int = 128
    shots: int = 0                # 0 -- весь train; иначе примеров на класс


def build_head(config: ProbeConfig) -> nn.Module:
    """Собрать голову probe поверх замороженных эмбеддингов размерности EMBEDDING_DIM."""
    if config.head == "linear":
        return nn.Linear(EMBEDDING_DIM, NUM_CLASSES)
    if config.head == "mlp":
        # TODO (задание 4.1): соберите non-linear probe:
        # Linear(EMBEDDING_DIM, hidden_dim) -> ReLU -> Dropout(dropout)
        # -> Linear(hidden_dim, NUM_CLASSES).
        raise NotImplementedError
    raise ValueError(f"неизвестный тип головы: {config.head}")


def few_shot_indices(labels: np.ndarray, shots: int, seed: int = SEED) -> np.ndarray:
    """Отобрать по shots примеров каждого класса (для few-shot-серии)."""
    rng = np.random.default_rng(seed)
    chosen = []
    for class_id in range(NUM_CLASSES):
        pool = np.flatnonzero(labels == class_id)
        rng.shuffle(pool)
        chosen.extend(pool[:shots])
    return np.asarray(chosen)


def train_probe(config: ProbeConfig) -> dict:
    """Обучить probe по единому протоколу и оценить его.

    Контракт:
        * фиксирует seed перед созданием головы;
        * при config.shots > 0 обучается только на few_shot_indices;
        * после каждой эпохи считает accuracy на VALIDATION;
        * запоминает состояние с лучшей validation accuracy (state_dict);
        * восстанавливает лучшее состояние и оценивает его на TEST один раз;
        * возвращает dict с val_accuracy, accuracy, macro_f1,
          trainable_params, train_time_s и историей по эпохам;
        * записывает результат через log_run.
    """
    raise NotImplementedError


# TODO (задание 4.2): реализуйте train_probe.
# TODO (задание 4.3): обучите linear probe на полном train и сравните
# с лучшим zero-shot. Прокомментируйте разницу.

linear_config = ProbeConfig(name="linear_probe_full", head="linear")
linear_config

#### 2.6 Non-linear probe

Non-linear probe добавляет один скрытый слой. Число обучаемых параметров растёт с $D \cdot C$ примерно до $D \cdot H + H \cdot C$; при $D = 512$, $H = 512$, $C = 10$ это рост более чем в 50 раз при том же объёме данных. Поэтому переобучение здесь — реальный риск, и его нужно диагностировать по расхождению train и validation, а не только по итоговой accuracy.

Сравнение корректно только при неизменном протоколе обучения: меняется единственный фактор — тип головы.

In [ ]:
mlp_config = ProbeConfig(name="mlp_probe_full", head="mlp",
                         hidden_dim=512, dropout=0.2)

# TODO (задание 5.1): обучите non-linear probe с тем же числом эпох,
# learning rate и weight decay, что и linear probe.
# TODO (задание 5.2): постройте кривые train loss и validation accuracy
# для обеих голов на одной фигуре и найдите эпоху, с которой начинается
# расхождение (признак переобучения).
# TODO (задание 5.3): ответьте в отчёте, что означает отсутствие прироста
# у non-linear probe относительно linear probe с точки зрения свойств
# признакового пространства CLIP.
# TODO (задание 5.4) (необязательно): проверьте влияние weight_decay
# на non-linear probe -- 2-3 значения, выбор по validation.

#### 2.7 Few-shot: сколько разметки нужно

Постройте зависимость качества linear probe от числа размеченных примеров на класс $k \in \{1, 2, 5, 10, 20, 40\}$ и нанесите на тот же график горизонтальную линию zero-shot.

Точка пересечения — количественный ответ на вопрос «с какого объёма разметки обучение окупается». При $k \le 5$ результат сильно зависит от того, какие именно примеры попали в выборку, поэтому каждую точку следует усреднить по 3 значениям seed и показать разброс. Без этого кривая недостоверна.

In [ ]:
SHOTS_GRID = [1, 2, 5, 10, 20, 40]
FEW_SHOT_SEEDS = [SEED, SEED + 1, SEED + 2]

# TODO (задание 6.1): для каждого k из SHOTS_GRID и каждого seed из FEW_SHOT_SEEDS
# обучите linear probe (протокол не меняется) и сохраните test accuracy.
# Все остальные параметры ProbeConfig совпадают с linear_config.

# TODO (задание 6.2): постройте график: по оси X -- k (логарифмическая шкала),
# по оси Y -- средняя test accuracy, с областью разброса (min-max или +-std
# по трём seed). Нанесите горизонтальную линию zero-shot.

# TODO (задание 6.3): определите наименьшее k, при котором linear probe
# устойчиво превосходит zero-shot (среднее минус разброс выше линии zero-shot).
# Сформулируйте вывод как утверждение о ДАННОЙ задаче и ДАННОМ энкодере,
# а не как универсальное правило.

#### 2.8 Сопоставимое сравнение

Сведите все стратегии в одну таблицу. Условия, одинаковые для всех строк: набор классов, разбиение, энкодер изображений и его препроцессинг, seed, устройство. Различие — только стратегия.

Обязательный столбец — **число обучаемых параметров**: он делает наглядной цену перехода от zero-shot (ноль параметров) к probe. Второй обязательный — время: у probe оно распадается на разовое извлечение эмбеддингов и обучение головы, и эти величины нужно приводить раздельно, иначе стоимость probe выглядит нереалистично низкой.

In [ ]:
SUMMARY_COLUMNS = [
    "name", "strategy", "prompt_set", "head", "shots",
    "trainable_params", "feature_time_s", "train_time_s",
    "val_accuracy", "accuracy", "macro_f1",
]

summary = results_table(SUMMARY_COLUMNS)
summary

# TODO (задание 7.1): дополните таблицу столбцом «полное время подготовки решения»
# = feature_time_s (train + val) + train_time_s. Для zero-shot это время
# построения текстовых эмбеддингов.
# TODO (задание 7.2): постройте столбчатую диаграмму accuracy по стратегиям
# с подписью числа обучаемых параметров над каждым столбцом.
# TODO (задание 7.3): проверьте согласованность ранжирования по validation и test.

#### 2.9 Анализ ошибок

Ошибки zero-shot и probe имеют разную природу. Zero-shot ошибается там, где текстовое описание класса неточно или неоднозначно; probe — там, где классы плохо разделимы в признаковом пространстве. Сопоставление их confusion matrix показывает, какая часть ошибок связана с формулировкой запроса, а какая — с самим представлением.

In [ ]:
def plot_confusion(y_true, y_pred, title: str) -> None:
    matrix = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    figure, axis = plt.subplots(figsize=(7, 6))
    image = axis.imshow(matrix, cmap="Blues")
    axis.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=90)
    axis.set_yticks(range(NUM_CLASSES), CLASS_NAMES)
    axis.set_xlabel("предсказание")
    axis.set_ylabel("истина")
    axis.set_title(title)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            axis.text(j, i, int(matrix[i, j]), ha="center", va="center", fontsize=8)
    figure.colorbar(image, ax=axis)
    plt.tight_layout()
    plt.show()


# TODO (задание 8.1): постройте confusion matrix для лучшего zero-shot
# и для linear probe на test.
# TODO (задание 8.2): выделите классы, где probe даёт наибольший прирост
# относительно zero-shot, и объясните причину через формулировку запроса.
# TODO (задание 8.3): найдите объекты, ошибочные у обеих стратегий, и покажите
# 4-6 таких изображений. Общие ошибки указывают на ограничение самого
# представления CLIP, а не стратегии его применения.
# TODO (задание 8.4): проверьте связь уверенности и правильности: постройте
# распределение максимальной вероятности для верных и неверных предсказаний.
# Обсудите, можно ли на его основе задать порог отказа.

### Отчёт

Отчётная часть включает:

1. **Условия эксперимента:** окружение и версии, seed, чекпоинт CLIP, список из 10 классов, значения `N_SUBSET`, `N_VAL`, `N_TEST`, размерность эмбеддингов.
2. **Полный список использованных текстовых шаблонов** для каждого набора. Zero-shot без указания шаблонов невоспроизводим.
3. **Сводную таблицу стратегий:** zero-shot (по лучшему набору шаблонов), linear probe, non-linear probe, few-shot-точки — с accuracy, macro F1, числом обучаемых параметров, временем извлечения эмбеддингов и временем обучения головы.
4. **Few-shot-кривую** с разбросом по трём seed и линией zero-shot; указанное наименьшее $k$, при котором probe устойчиво превосходит zero-shot.
5. **Кривые обучения** linear и non-linear probe с комментарием о переобучении.
6. **Две confusion matrix** (zero-shot и probe) и разбор классов с наибольшей разницей.
7. **Правило выбора стратегии** для данной задачи: при каком бюджете разметки какую стратегию следует применять и почему.
8. **Выводы, отделённые от наблюдений**, с ограничениями: один энкодер, один датасет, подвыборка `N_SUBSET`, ограниченный набор шаблонов, отсутствие сравнения с полным fine-tuning.

Работа не засчитывается, если: набор шаблонов или гиперпараметры probe выбирались по test; probe-конфигурации обучались по разным протоколам; few-shot-кривая построена по одному seed без указания разброса; приведён только лучший результат без журнала `runs.jsonl`.

### Контрольные вопросы

1. Что оптимизирует контрастная функция потерь CLIP и какую роль в ней играет температура $\tau$?
2. Почему CLIP позволяет классифицировать объекты классов, которых не было в явной разметке обучения?
3. Почему эмбеддинги перед вычислением косинусной близости обязательно нормируются?
4. В чём смысл ансамблирования текстовых шаблонов и почему усреднять нужно уже нормированные векторы, а результат нормировать повторно?
5. Почему подбор набора шаблонов по тестовой выборке делает оценку zero-shot недостоверной, хотя обучаемых параметров нет?
6. Что именно измеряет linear probe и что означает его высокое качество применительно к признаковому пространству?
7. О чём говорит отсутствие прироста у non-linear probe при кратно большем числе параметров?
8. Чем probe отличается от fine-tuning по стоимости, риску переобучения и требуемому объёму разметки?
9. Как интерпретировать точку пересечения few-shot-кривой с уровнем zero-shot и почему это не универсальная константа?
10. По каким признакам в анализе ошибок можно отличить ограничение формулировки запроса от ограничения самого представления?